# Graph Sparsification Experiments

Generate graphs (Configuration Model + wSBM), compute Metric Backbone and Effective Resistance
sparsifications, run SIR simulations on all, and compare infection probabilities.

In [ ]:
import numpy as np
from scipy import sparse
import matplotlib.pyplot as plt
import time

from graph_sparsification.generators import configuration_model, wsbm_fast
from graph_sparsification.sparsifiers import metric_backbone, effective_resistance_sparsify
from graph_sparsification.sir import sir_monte_carlo
from graph_sparsification.visualization import (
    plot_adjacency_comparison,
    plot_multi_infection_comparison,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

## 1. Helper functions

In [ ]:
def run_experiment(W, name, communities=None, beta=0.3, gamma=1.0,
                   initial_infected=None, n_runs=200, effr_fraction=None, rng=None):
    """Run full experiment: sparsify + SIR on original and sparsified graphs."""
    rng = np.random.default_rng(rng)
    n = W.shape[0]
    n_edges_orig = sparse.triu(W).nnz
    
    if initial_infected is None:
        # Pick highest-degree node
        degrees = np.array(W.sum(axis=1)).ravel()
        initial_infected = [np.argmax(degrees)]
    
    print(f"\n{'='*60}")
    print(f"Experiment: {name}")
    print(f"  n={n}, edges={n_edges_orig}")
    print(f"  beta={beta}, gamma={gamma}, n_runs={n_runs}")
    print(f"{'='*60}")
    
    # --- Metric Backbone ---
    t0 = time.time()
    W_mbb = metric_backbone(W)
    t_mbb = time.time() - t0
    n_edges_mbb = sparse.triu(W_mbb).nnz
    print(f"  MBB: {n_edges_mbb} edges ({n_edges_mbb/n_edges_orig*100:.1f}%) [{t_mbb:.2f}s]")
    
    # --- Effective Resistance ---
    # Target same number of edges as MBB for fair comparison
    if effr_fraction is None:
        effr_fraction = max(n_edges_mbb / n_edges_orig, 0.05)
    t0 = time.time()
    W_effr = effective_resistance_sparsify(W, fraction=effr_fraction, rng=rng)
    t_effr = time.time() - t0
    n_edges_effr = sparse.triu(W_effr).nnz
    print(f"  EffR: {n_edges_effr} edges ({n_edges_effr/n_edges_orig*100:.1f}%) [{t_effr:.2f}s]")
    
    # --- SIR simulations ---
    print(f"  Running SIR ({n_runs} runs each)...")
    
    t0 = time.time()
    sir_orig = sir_monte_carlo(W, beta, gamma, initial_infected, n_runs=n_runs, rng=rng)
    print(f"    Original: {time.time()-t0:.2f}s")
    
    t0 = time.time()
    sir_mbb = sir_monte_carlo(W_mbb, beta, gamma, initial_infected, n_runs=n_runs, rng=rng)
    print(f"    MBB:      {time.time()-t0:.2f}s")
    
    t0 = time.time()
    sir_effr = sir_monte_carlo(W_effr, beta, gamma, initial_infected, n_runs=n_runs, rng=rng)
    print(f"    EffR:     {time.time()-t0:.2f}s")
    
    # --- Adjacency matrix comparison ---
    fig1 = plot_adjacency_comparison(W, W_mbb,
                                     labels=(f"{name} Original", f"{name} MBB"),
                                     communities=communities)
    plt.show()
    
    fig1b = plot_adjacency_comparison(W, W_effr,
                                      labels=(f"{name} Original", f"{name} EffR"),
                                      communities=communities)
    plt.show()
    
    # --- Infection probability comparison ---
    fig2 = plot_multi_infection_comparison(
        sir_orig['infection_prob'],
        [sir_mbb['infection_prob'], sir_effr['infection_prob']],
        ['Metric Backbone', 'Effective Resistance'],
    )
    plt.suptitle(f"{name}: Infection Probability Comparison", fontsize=14, y=1.02)
    plt.show()
    
    return {
        'W': W, 'W_mbb': W_mbb, 'W_effr': W_effr,
        'sir_orig': sir_orig, 'sir_mbb': sir_mbb, 'sir_effr': sir_effr,
    }

## 2. Experiment 1: Configuration Model

In [ ]:
# Configuration model: n=200 nodes, avg degree ~10, exponential weights
rng = np.random.default_rng(42)
n = 200
degrees = rng.poisson(10, size=n)
degrees = np.maximum(degrees, 1)  # ensure minimum degree 1

W_cm = configuration_model(degrees, weight_distribution="exponential",
                           weight_params={"scale": 1.0}, rng=42)

print(f"Configuration Model: n={n}, edges={sparse.triu(W_cm).nnz}")
print(f"Avg weighted degree: {np.array(W_cm.sum(axis=1)).ravel().mean():.2f}")

In [ ]:
results_cm = run_experiment(W_cm, "Configuration Model", beta=0.3, gamma=1.0,
                            n_runs=200, rng=42)

## 3. Experiment 2: wSBM with 3 communities

In [ ]:
# wSBM: 3 communities, strong intra-community connections
n = 300
k = 3
pi = np.array([1/3, 1/3, 1/3])

# B matrix: high intra, low inter
B = np.array([
    [12, 2, 2],
    [2, 12, 2],
    [2, 2, 12],
], dtype=float)

# Weight rates: intra-community edges have smaller costs (closer)
Lambda = np.array([
    [2.0, 0.5, 0.5],
    [0.5, 2.0, 0.5],
    [0.5, 0.5, 2.0],
])

W_sbm, z_sbm = wsbm_fast(n, k, pi, B, weight_distribution="exponential",
                          Lambda=Lambda, rng=42)

print(f"wSBM: n={n}, k={k}, edges={sparse.triu(W_sbm).nnz}")
for c in range(k):
    print(f"  Community {c}: {(z_sbm == c).sum()} nodes")

In [ ]:
results_sbm3 = run_experiment(W_sbm, "wSBM (k=3)", communities=z_sbm,
                              beta=0.5, gamma=1.0, n_runs=200, rng=42)

## 4. Experiment 3: wSBM with 2 communities (planted partition)

In [ ]:
# Planted partition: 2 balanced communities
n = 250
k = 2
pi = np.array([0.5, 0.5])

B = np.array([
    [15, 3],
    [3, 15],
], dtype=float)

Lambda = np.array([
    [3.0, 0.8],
    [0.8, 3.0],
])

W_pp, z_pp = wsbm_fast(n, k, pi, B, weight_distribution="exponential",
                        Lambda=Lambda, rng=123)

print(f"Planted Partition: n={n}, k={k}, edges={sparse.triu(W_pp).nnz}")
for c in range(k):
    print(f"  Community {c}: {(z_pp == c).sum()} nodes")

In [ ]:
results_pp = run_experiment(W_pp, "Planted Partition (k=2)", communities=z_pp,
                            beta=0.4, gamma=1.0, n_runs=200, rng=123)

## 5. Experiment 4: Dense Configuration Model (high degree)

In [ ]:
# Denser graph to see more dramatic sparsification
n = 150
degrees = rng.poisson(25, size=n)
degrees = np.maximum(degrees, 2)

W_dense = configuration_model(degrees, weight_distribution="uniform",
                              weight_params={"low": 0.1, "high": 2.0}, rng=99)

print(f"Dense CM: n={n}, edges={sparse.triu(W_dense).nnz}")
print(f"Avg weighted degree: {np.array(W_dense.sum(axis=1)).ravel().mean():.2f}")

In [ ]:
results_dense = run_experiment(W_dense, "Dense Config Model", beta=0.15, gamma=1.0,
                               n_runs=200, rng=99)

## 6. Summary: R² comparison across experiments

In [ ]:
def compute_r2(p_orig, p_sparse):
    mask = np.isfinite(p_orig) & np.isfinite(p_sparse)
    po, ps = p_orig[mask], p_sparse[mask]
    ss_res = np.sum((po - ps) ** 2)
    ss_tot = np.sum((po - po.mean()) ** 2)
    return 1 - ss_res / ss_tot if ss_tot > 0 else 0.0

experiments = [
    ("Config Model", results_cm),
    ("wSBM (k=3)", results_sbm3),
    ("Planted Partition", results_pp),
    ("Dense CM", results_dense),
]

print(f"{'Experiment':<22} {'MBB R²':>10} {'EffR R²':>10} {'MBB edges%':>12} {'EffR edges%':>12}")
print("-" * 70)
for name, res in experiments:
    r2_mbb = compute_r2(res['sir_orig']['infection_prob'], res['sir_mbb']['infection_prob'])
    r2_effr = compute_r2(res['sir_orig']['infection_prob'], res['sir_effr']['infection_prob'])
    e_orig = sparse.triu(res['W']).nnz
    e_mbb = sparse.triu(res['W_mbb']).nnz
    e_effr = sparse.triu(res['W_effr']).nnz
    print(f"{name:<22} {r2_mbb:>10.3f} {r2_effr:>10.3f} {e_mbb/e_orig*100:>11.1f}% {e_effr/e_orig*100:>11.1f}%")

In [ ]:
# Bar chart summary
names = [e[0] for e in experiments]
r2_mbb_vals = [compute_r2(e[1]['sir_orig']['infection_prob'], e[1]['sir_mbb']['infection_prob'])
               for e in experiments]
r2_effr_vals = [compute_r2(e[1]['sir_orig']['infection_prob'], e[1]['sir_effr']['infection_prob'])
                for e in experiments]

x = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, r2_mbb_vals, width, label='Metric Backbone', color='steelblue')
bars2 = ax.bar(x + width/2, r2_effr_vals, width, label='Effective Resistance', color='coral')

ax.set_ylabel('$R^2$ (Infection Probability)', fontsize=12)
ax.set_title('Sparsifier Fidelity: $R^2$ of Node Infection Probabilities', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=10)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.05)
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.3)

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

plt.tight_layout()
plt.show()